# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://github.com/ArnavP2305/flyrank-ml-internship-2/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

**1. What one row means (Unit of Analysis Grain):**  
One row represents a single **pseudonymized content page** (a content item) belonging to a specific client. In database terms, the unique grain is defined by the compound key `[client_hash_id, content_hash_id]`.

**2. Time Window / Split:**  
We will iterate on a mid-panel month partition: **March 2026** (`month=2026-03`).
- **Feature Window:** Trailing 30 days prior to March 1, 2026 (i.e. February 2026).
- **Target/Outcome Window:** The month of March 2026 itself.

**3. Gated Tables Used:**  
- `dim_clients` (client access start dates)
- `dim_content` (metadata and content age)
- `fact_content_daily_performance` (partitioned by month; primary time series)
- `fact_content_query_90d` (query concentration and search profile metrics)

**4. Target to Predict:**  
We predict `is_declining` (a binary label where `1` represents organic GSC impressions dropping by more than 20% compared to the previous month, and `0` represents stable or growing pages).

**5. Deliberately Excluded:**  
- `trend_pct` and `trend_direction` from the target window (causes immediate label leakage).
- Raw client IDs or content IDs as features (avoids memorizing specific client behaviors).

In [1]:
# Helper: Load HF token from local .env or environment
import os, sys
import numpy as np, pandas as pd
env_path = '.env'
for _ in range(4):
    if os.path.exists(env_path):
        with open(env_path) as f:
            for line in f:
                if line.strip() and not line.startswith('#'):
                    k, v = line.strip().split('=', 1)
                    os.environ[k] = v
        break
    env_path = os.path.join('..', env_path)

HF_TOKEN = os.environ.get('HF_TOKEN')
assert HF_TOKEN is not None, "HF_TOKEN environment variable not set!"

## 2. Fields: feature / label / context / excluded

Every field is categorized strictly into one of the four buckets:

### 1. Features (Safe & knowable before March 1, 2026)
- `imp_prev30`: Total GSC impressions in the trailing 30 days (Feb 2026). Knowable historically.
- `clk_prev30`: Total GSC clicks in the trailing 30 days. Knowable historically.
- `pos_prev30`: Average GSC rank position in the trailing 30 days. Knowable historically.
- `content_age_days`: Elapsed days since content creation. Knowable historically.
- `days_since_last_update`: Days since last content refresh/update. Knowable historically.
- `visible_queries`: Number of distinct search queries the page ranked for in the trailing 90 days. Knowable historically.

### 2. Label / Proxy (The outcome measured in March 2026)
- `is_declining`: Binary indicator where GSC impressions in the target window (March 2026) drops by > 20% compared to the feature window (Feb 2026).

### 3. Context (For joins, splits, and grouping — never fed to model)
- `client_hash_id`: Client pseudonym. Used for client-holdout train/test splits.
- `content_hash_id`: Unique page identifier. Used for joining and aggregation.

### 4. Excluded (Why excluded)
- `imp_last30`: Total GSC impressions in March 2026 (excluded because it constitutes the future outcome we want to predict).
- `trend_pct` / `trend_direction` (March 2026): Excluded because they are direct derivations of the target label (leakage).

In [2]:
# Connect to DuckDB and configure the Hugging Face secret
import duckdb
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {
    'dim_clients': f"read_parquet('{REL}/dim_clients.parquet')",
    'dim_content': f"read_parquet('{REL}/dim_content.parquet')",
    'fact_daily':  f"read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')",
    'fact_query_90d': f"read_parquet('{REL}/fact_content_query_90d.parquet')"
}
print('DuckDB connected successfully.')

DuckDB connected successfully.


## 3. Verify it with queries (grain, counts, missing values, windows)

We run three verification queries on the `month=2026-03` partition:

1. **Query 1 (Grain verification):** Group by the key `[client_hash_id, content_hash_id]` on the aggregated monthly dataset and ensure no duplicate records exist.
2. **Query 2 (Row count & dates span):** Show the exact row count and min/max report dates in our slice.
3. **Query 3 (Availability check):** Filter GSC metrics and GA4 metrics with `IS TRUE` checks and show how many rows survive.

In [3]:
# --- Query 1: Verify the Grain (should return 0 rows) ---
print('=== Query 1: Grain Duplicate Check ===')
dup_check = con.sql(f"""
    SELECT client_hash_id, content_hash_id, COUNT(*)
    FROM {TABLES['fact_daily']}
    GROUP BY client_hash_id, content_hash_id, report_date
    HAVING COUNT(*) > 1
    LIMIT 5
""").df()
print(f'Duplicates at (client, content, date) level: {len(dup_check)}')
assert len(dup_check) == 0, "Duplicate entries found!"

# --- Query 2: Row Count and Date Span of March 2026 partition ---
print('\n=== Query 2: Partition Row Count & Date Span ===')
partition_stats = con.sql(f"""
    SELECT COUNT(*) as row_count,
           MIN(report_date) as start_date,
           MAX(report_date) as end_date
    FROM {TABLES['fact_daily']}
""").df()
print(partition_stats.to_string(index=False))

# --- Query 3: Availability check (how many rows survive IS TRUE) ---
print('\n=== Query 3: GA4 Data Availability Filter ===')
availability_stats = con.sql(f"""
    SELECT 
        COUNT(*) as total_rows,
        SUM(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END) as ga4_available_rows,
        ROUND(100.0 * SUM(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END) / COUNT(*), 2) as ga4_available_pct
    FROM {TABLES['fact_daily']}
""").df()
print(availability_stats.to_string(index=False))

=== Query 1: Grain Duplicate Check ===


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Duplicates at (client, content, date) level: 0

=== Query 2: Partition Row Count & Date Span ===


 row_count start_date   end_date
   9841378 2026-03-01 2026-03-31

=== Query 3: GA4 Data Availability Filter ===


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

 total_rows  ga4_available_rows  ga4_available_pct
    9841378            413966.0               4.21


### 3.2 Feature Building & The Leakage Trap
We now assemble our **5 features** for March 2026, building them purely from the historical (pre-March 1, 2026) panel dates to prevent leakage. 

We will intentionally perform the **Leakage Trap Experiment**:
1. Train a model with honest features → compute Precision@50.
2. Inject a leaky feature (`imp_last30` which represents the actual outcome target volume) → see the metric jump to a perfect 1.000.
3. Remove the leaky feature → restore and verify the honest, realistic performance metric.

In [4]:
# Helper evaluation function
def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    topk = np.asarray(labels)[order[:k]]
    return topk.mean()

# 1. Build a clean feature frame using Feb 2026 as the feature window, and March 2026 as target
print('Building features in DuckDB...')
raw_data = con.sql(f"""
    WITH feb_features AS (
        SELECT client_hash_id, content_hash_id,
               SUM(gsc_impressions) AS imp_prev30,
               SUM(gsc_clicks) AS clk_prev30,
               AVG(gsc_avg_position) AS pos_prev30
        FROM read_parquet('{REL}/fact_content_daily_performance/month=2026-02/*.parquet')
        GROUP BY 1, 2
    ),
    march_labels AS (
        SELECT client_hash_id, content_hash_id,
               SUM(gsc_impressions) AS imp_last30
        FROM {TABLES['fact_daily']}
        GROUP BY 1, 2
    )
    SELECT f.client_hash_id, f.content_hash_id,
           f.imp_prev30, f.clk_prev30, f.pos_prev30,
           date_diff('day', c.content_created_date, DATE '2026-03-01') AS content_age_days,
           date_diff('day', c.content_updated_date, DATE '2026-03-01') AS days_since_last_update,
           l.imp_last30
    FROM feb_features f
    JOIN march_labels l ON f.client_hash_id = l.client_hash_id AND f.content_hash_id = l.content_hash_id
    LEFT JOIN {TABLES['dim_content']} c ON f.content_hash_id = c.content_hash_id
    WHERE f.imp_prev30 >= 100
""").df()

raw_data['is_declining'] = (raw_data['imp_last30'] < 0.8 * raw_data['imp_prev30']).astype(int)
data_clean = raw_data.dropna(subset=['content_age_days', 'days_since_last_update', 'pos_prev30']).copy()
print(f'Engineered dataset size: {data_clean.shape[0]:,} rows')

# Define the 5 honest features
honest_features = ['imp_prev30', 'clk_prev30', 'pos_prev30', 'content_age_days', 'days_since_last_update']
X_honest = data_clean[honest_features]
y = data_clean['is_declining']

# Fit model with honest features
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
X_tr, X_te, y_tr, y_te = train_test_split(X_honest, y, test_size=0.3, random_state=42, stratify=y)
rf_honest = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1).fit(X_tr, y_tr)
p50_honest = precision_at_k(rf_honest.predict_proba(X_te)[:, 1], y_te, 50)
print(f'Honest Model Precision@50: {p50_honest:.4f}')

# --- THE LEAKAGE TRAP EXPERIMENT ---
# We inject imp_last30 directly into features. This column represents future outcome volume.
print('\n=== INJECTING LEAKAGE TRAP ===')
leaky_features = honest_features + ['imp_last30']
X_leaky = data_clean[leaky_features]
X_tr_l, X_te_l, y_tr_l, y_te_l = train_test_split(X_leaky, y, test_size=0.3, random_state=42, stratify=y)
rf_leaky = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1).fit(X_tr_l, y_tr_l)
p50_leaky = precision_at_k(rf_leaky.predict_proba(X_te_l)[:, 1], y_te_l, 50)
print(f'Leaky Model Precision@50: {p50_leaky:.4f}  <-- Artificial perfection!')
print('The model simply memorized that pages with lower imp_last30 are declining.')

# --- REMOVE LEAKAGE ---
print('\n=== REMOVING LEAKAGE ===')
print(f'Restored clean feature columns: {honest_features}')
print(f'Confirmed final verified Precision@50: {p50_honest:.4f}')

Building features in DuckDB...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Engineered dataset size: 76,837 rows


Honest Model Precision@50: 0.8600

=== INJECTING LEAKAGE TRAP ===


Leaky Model Precision@50: 1.0000  <-- Artificial perfection!
The model simply memorized that pages with lower imp_last30 are declining.

=== REMOVING LEAKAGE ===
Restored clean feature columns: ['imp_prev30', 'clk_prev30', 'pos_prev30', 'content_age_days', 'days_since_last_update']
Confirmed final verified Precision@50: 0.8600


## 4. Data limits

### Major Limitation of the Warehouse Dataset: Unbalanced Panel Depth
The dataset is an **unbalanced panel** — meaning different clients have varying history starting points. If we train a model using feature windows that require 60 or 90 days of history, we will automatically exclude or bias the data for clients who recently signed up. Let us prove this by showing how GSC tracking start dates vary across clients.

In [5]:
client_start_dates = con.sql(f"""
    SELECT client_hash_id, gsc_data_start, ga4_data_start
    FROM {TABLES['dim_clients']}
    ORDER BY gsc_data_start ASC
""").df()

print('Client Search Console start date distribution:')
print(client_start_dates['gsc_data_start'].describe())
print(f'\nClients with Ga4 data start: {client_start_dates["ga4_data_start"].notnull().sum()} / {len(client_start_dates)}')

Client Search Console start date distribution:
count                            67
mean     2025-11-17 00:42:59.104477
min             2025-01-27 00:00:00
25%             2025-09-24 00:00:00
50%             2025-11-05 00:00:00
75%             2026-02-19 00:00:00
max             2026-06-02 00:00:00
Name: gsc_data_start, dtype: object

Clients with Ga4 data start: 51 / 104


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.